# Auditoría del estudio automotor
Ejecutar desde la raíz del tablero o docs/analysis. Usa biblioteca estándar. Los originales en output/automotor no se distribuyen públicamente; si faltan, el contador lo informa. Revisar las condiciones comerciales en el JSON y el informe HTML.

In [1]:
import json, hashlib
from pathlib import Path
root = Path.cwd()
if not (root / "src/data/automotive-study.json").exists(): root = root.parent.parent
s = json.loads((root / "src/data/automotive-study.json").read_text(encoding="utf-8"))
total = s["fleet"]["annual"][-1]["value"]
for key in ["departments", "classes", "ages"]:
    assert sum(r["value"] for r in s["fleet"][key]) == total
ids = {o["id"] for o in s["offers"]}
assert len(ids) == len(s["offers"])
for p in s["comparisons"]:
    assert p["boliviaId"] in ids and p["foreignId"] in ids
    assert p["grade"] == "A" or not p["directGapAllowed"]
print("Parque reconciliado:", total)
print("Ofertas:", len(ids), "Pares:", len(s["comparisons"]))
for heading in ["8702","8703","8704"]:
    rows = {r["year"]:r for r in s["trade"]["annual"] if r["heading"] == heading}
    assert len(rows[2024]["months"]) == len(rows[2025]["months"]) == 12
    print(heading, "CIF 2025:", rows[2025]["usd"], "Cambio %:", round((rows[2025]["usd"] / rows[2024]["usd"] - 1)*100, 3))
verified = 0
for source in s["sources"]:
    file = root / source["archive"]
    if file.exists():
        assert hashlib.sha256(file.read_bytes()).hexdigest() == source["sha256"]
        verified += 1
print("Archivos locales con hash verificado:", verified, "de", len(s["sources"]))


Parque reconciliado: 2672176
Ofertas: 72 Pares: 8
8702 CIF 2025: 39067393 Cambio %: -0.153
8703 CIF 2025: 286593687 Cambio %: -15.173
8704 CIF 2025: 137639291 Cambio %: 1.438
Archivos locales con hash verificado: 40 de 40
